# Arm G re-extraction — a direction that reads scope and not catalog position

Section 14 settled two things at once: the decision is **entirely** catalog
position (64/64 conflict decisions reverse on a two-line swap), and the scope
signal is nonetheless real (condition main effect +4.814, within-order label
AUROC 1.00000). So there is a variable worth having a direction for, and the
direction we have is not it — it separates catalog order within the conflict
condition at **AUROC 1.0000** against **0.9431** for the label.

Write the state as `h(C, O) = mu + a*C + b*O + d*C*O`, with C and O in {-1, +1}.

The within-scenario paired difference cancels the order main effect already,
since both members of a pair share a rendering: `h(+1,o) - h(-1,o) = 2a + 2d*o`.

**Averaging that over both orders is not the fix**, which is what the self-test
caught. The parity-locked assignment is *balanced* across pairs, so mean(o) = 0
and the interaction cancels in the legacy mean too. The old estimate of `a` was
never biased.

The contamination is at the **projection** level. Projecting onto `a` picks up
the interaction whenever `a` and `d` are not orthogonal *in the model's
geometry*, however cleanly `a` was estimated. So the fix is explicit
orthogonalization — and crossed source data is what makes it possible, because
without both renderings of a scenario `d` is not estimable at all.

Five directions from the same crossed source states, so the recipe is isolated
from the data:

| recipe | is |
|---|---|
| `order_averaged` | mean paired difference over both orders (a) |
| `legacy_recipe` | one order per pair, as the old generator forced (a, noisier) |
| `order_main` | mean of h(., outside) − h(., inside) (b) |
| `order_difference` | paired diff at outside minus at inside (d) |
| `orthogonalized` | `a` with `b` and `d` projected out — **the fix** |

Each is scored on held-out crossed data by label AUROC versus how well it reads
catalog order *within* a condition. Baseline capture only — no intervention.

In [ ]:
print("Protocol: ARM_G_REEXTRACT_V1")
%pip -q install "transformers==5.0.0" "accelerate==1.12.0" \
  "sentence-transformers==5.2.2" "scikit-learn==1.8.0"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, shutil
LAUNCH_DIR = "/content/drive/MyDrive/phi-map/arm-g-reextract-launch"
LAUNCH_FILES = (
    "arm_g_reextract.py",
    "arm_g_order_crossover.py",
    "arm_g_cross_layer.py",
    "arm_g_causal_subspace.py",
    "arm_g_causal_dose_ablation.py",
    "arm_g_causal.py",
    "arm_g_phase1.py",
    "arm_g_scenarios.py",
)
for name in LAUNCH_FILES:
    source = f"{LAUNCH_DIR}/{name}"
    assert os.path.exists(source), f"Missing {source}"
    shutil.copy2(source, f"/content/{name}")
print("Arm G re-extraction launch files staged: OK")

In [ ]:
ACTING_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
WORK_DIR = "/content/drive/MyDrive/phi-map/arm-g-reextract-seed112-v1"
PAIRS_PER_FAMILY = 16
SOURCE_PAIRS_PER_FAMILY = 48
BATCH_SIZE = 16

from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
assert HF_TOKEN, "Add an HF_TOKEN secret with Llama-3.1-8B-Instruct access"

In [ ]:
from huggingface_hub import hf_hub_download
hf_hub_download(ACTING_MODEL, "config.json", token=HF_TOKEN)
print("Hugging Face model access: OK")

import subprocess, torch
subprocess.run(["nvidia-smi"], check=True)
props = torch.cuda.get_device_properties(0)
print(props.name, round(props.total_memory / 1024**3, 1), "GiB")
assert "A100" in props.name and props.total_memory >= 35 * 1024**3
assert torch.cuda.is_bf16_supported()

In [ ]:
# Deterministic tests, no GPU and no network:
#  - both estimators of the condition axis are unbiased (200-replication
#    Monte Carlo on the component orthogonal to the condition axis)
#  - with cos(a,d)=0.8 both are contaminated at the projection level anyway,
#    and orthogonalization repairs it while RAISING label AUROC
#  - the contamination bar rejects a genuinely clean direction ~2.5% of the time
import os, subprocess, sys
env = dict(os.environ)
env["HF_TOKEN"] = HF_TOKEN
base_cmd = [
    sys.executable, "/content/arm_g_reextract.py",
    "--model", ACTING_MODEL,
    "--output-dir", WORK_DIR,
    "--pairs-per-family", str(PAIRS_PER_FAMILY),
    "--source-pairs-per-family", str(SOURCE_PAIRS_PER_FAMILY),
    "--batch-size", str(BATCH_SIZE),
]
subprocess.run(base_cmd + ["--self-test"], check=True, env=env)

In [ ]:
# Capture states across the layer sweep and score all five recipes.
subprocess.run(base_cmd, check=True, env=env)

In [ ]:
import json
result_path = f"{WORK_DIR}/arm_g_reextract_result.json"
result = json.load(open(result_path))
print("decision:", result["decision"])
for reason in result["decision_reasons"]:
    print("  -", reason)
print("selected layer:", result["selected_layer"])
print()
hdr = f"{'layer':>5}  {'recipe':18s} {'label':>7} {'order|confl':>12} {'order|reach':>12} {'contam':>7}  clean"
print(hdr); print("-" * len(hdr))
for layer, entry in result["layer_profile"].items():
    for recipe, e in entry.items():
        w = e["within_condition"]
        print(f"{layer:>5}  {recipe:18s} {e['label_auroc']:7.4f} "
              f"{w['conflict']['order_auroc']:12.4f} {w['reachable']['order_auroc']:12.4f} "
              f"{e['order_contamination']:7.4f}  {e['clean']}")
    print()

In [ ]:
# The geometry is the finding: how far is the condition axis from the
# interaction axis, layer by layer? That angle is what the old direction was
# silently reading.
print(f"{'layer':>5} {'cos(a, d)':>10} {'cos(a, b)':>10} {'cos(a, orth)':>13}")
for layer, cos in result["direction_cosines"].items():
    print(f"{layer:>5} {cos['order_averaged_vs_order_difference']:>10.4f} "
          f"{cos['order_averaged_vs_order_main']:>10.4f} "
          f"{cos['order_averaged_vs_orthogonalized']:>13.4f}")

In [ ]:
import base64, gzip, json

summary = {k: result[k] for k in
           ("decision", "decision_reasons", "selected_layer", "layer_profile",
            "direction_cosines", "prior_evidence", "sample_counts", "audits")}
summary_path = f"{WORK_DIR}/arm_g_reextract_result_summary.json"
archive_path = f"{WORK_DIR}/arm_g_reextract_result.json.gz.b64"
with open(summary_path, "w") as h:
    json.dump({**summary,
        "protocol": {"source_seeds": [101, 102], "evaluation_seed": 112,
        "catalog_order_mode": "crossed", "control_label_mode": "parity_independent",
        "intervention": "none"},
        "full_result_artifact": "arm_g_reextract_result.json.gz.b64"}, h, indent=1)
with open(archive_path, "wb") as h:
    h.write(base64.b64encode(gzip.compress(json.dumps(result).encode("utf-8"))))
print("summary:", summary_path)
print("archive:", archive_path)